In [21]:
import pandas as pd
from faker import Faker
import random
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly_resampler import FigureResampler, register_plotly_resampler

In [2]:
fake = Faker()

fake_ip_v6 = fake.ipv6()
print(f'Fake IPv4: {fake_ip_v6}')

Fake IPv4: 7a83:abe0:decc:8cc7:adc6:1770:acfe:790c


In [6]:
data = []
for _ in range(10000000):
    sip = fake.ipv4()
    dip = fake.ipv4()
    sp = random.randint(1024, 65535)
    dp = random.randint(1024, 65535)
    timestamp = fake.date_time_this_year()
    protocol = random.choice(['TCP', 'UDP', 'ICMP'])
    oct = random.randint(64, 1500)

    data.append({
        "timestamp": timestamp,
        "sip": sip,
        "dip": dip,
        "sp": sp,
        "dp": dp,
        "protocol": protocol,
        "oct": oct,
    })
df = pd.DataFrame(data)
print(df.head())

In [3]:
df = pd.read_csv('pcap2ipfix-applabel-yafscii.csv')
df

,start-time,end-time,duration,rtt,proto,sip,sp,dip,dp,iflags,...,tag,rtag,pkt,oct,rpkt,roct,app,entropy,rentropy,end-reason
0,2021-09-01 16:08:02.819,2021-09-01 16:08:02.850,0.031,0.001,6,192.168.50.133,46250,192.168.50.237,8080,S,...,0,0,12,1190,9,1676,80,174,181,NaN
1,2021-09-01 16:08:02.852,2021-09-01 16:08:02.873,0.021,0.000,6,192.168.50.133,46252,192.168.50.237,8080,S,...,0,0,16,1606,15,10558,80,183,137,NaN
2,2021-09-01 16:08:10.357,2021-09-01 16:08:10.530,0.173,0.009,6,192.168.50.127,39768,192.168.50.50,443,S,...,0,0,34,3070,53,60944,443,229,247,NaN
3,2021-09-01 16:08:16.328,2021-09-01 16:08:16.328,0.000,0.000,6,192.168.60.159,59851,192.168.60.149,8002,S,...,0,0,2,104,1,40,0,0,0,NaN
4,2021-09-01 16:08:16.328,2021-09-01 16:08:16.328,0.000,0.000,6,192.168.60.149,8002,192.168.60.159,59851,AR,...,0,0,1,40,0,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12279,2021-09-01 20:16:31.852,2021-09-01 20:37:57.863,1286.011,0.002,6,192.168.50.248,21594,192.168.50.237,502,AP,...,0,0,20580,1008432,15434,833434,0,0,0,eof
12280,2021-09-01 20:16:31.774,2021-09-01 20:37:57.863,1286.089,0.048,6,192.168.50.248,49320,192.168.50.249,60330,AP,...,0,0,42176,5042662,29130,4337560,0,0,0,eof
12281,2021-09-01 20:16:32.419,2021-09-01 20:37:57.866,1285.447,0.000,1,192.168.50.254,0,192.168.50.50,781,0,...,0,0,1112,62272,0,0,0,0,0,eof
12282,2021-09-01 20:16:31.757,2021-09-01 20:37:57.868,1286.111,0.023,6,192.168.60.107,52202,192.168.60.101,102,AP,...,0,0,290570,17506816,167136,10172256,0,0,0,eof


In [4]:
df_copy = df.copy()

In [18]:
df['start-time'] = pd.to_datetime(df['start-time'])
df['end-time'] = pd.to_datetime(df['end-time'])
random_seconds = np.random.randint(1, 86401, size=len(df_copy))
random_offsets = pd.to_timedelta(random_seconds, unit='s')
df_copy['start-time'] = df['start-time'] + (df['end-time'].max() - df['start-time'].min()) + random_offsets
df_copy['end-time'] = df['end-time'] + (df['end-time'].max() - df['start-time'].min()) + random_offsets

In [16]:
random_values = np.random.randint(1, 11, size=len(df_copy))
df_copy['oct'] = df_copy['oct'] + random_values
df_copy['roct'] = df_copy['oct'] + random_values
df_copy['pkt'] = df_copy['oct'] + random_values
df_copy['rpkt'] = df_copy['oct'] + random_values

In [19]:
df_combined = pd.concat([df, df_copy])
df_combined.reset_index(drop=True, inplace=True)
df_combined

,start-time,end-time,duration,rtt,proto,sip,sp,dip,dp,iflags,...,tag,rtag,pkt,oct,rpkt,roct,app,entropy,rentropy,end-reason
0,2021-09-01 16:08:02.819,2021-09-01 16:08:02.850,0.031,0.001,6,192.168.50.133,46250,192.168.50.237,8080,S,...,0,0,12,1190,9,1676,80,174,181,NaN
1,2021-09-01 16:08:02.852,2021-09-01 16:08:02.873,0.021,0.000,6,192.168.50.133,46252,192.168.50.237,8080,S,...,0,0,16,1606,15,10558,80,183,137,NaN
2,2021-09-01 16:08:10.357,2021-09-01 16:08:10.530,0.173,0.009,6,192.168.50.127,39768,192.168.50.50,443,S,...,0,0,34,3070,53,60944,443,229,247,NaN
3,2021-09-01 16:08:16.328,2021-09-01 16:08:16.328,0.000,0.000,6,192.168.60.159,59851,192.168.60.149,8002,S,...,0,0,2,104,1,40,0,0,0,NaN
4,2021-09-01 16:08:16.328,2021-09-01 16:08:16.328,0.000,0.000,6,192.168.60.149,8002,192.168.60.159,59851,AR,...,0,0,1,40,0,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24563,2021-09-02 09:18:08.334,2021-09-02 09:39:34.345,1286.011,0.002,6,192.168.50.248,21594,192.168.50.237,502,AP,...,0,0,1008461,1008460,1008461,1008461,0,0,0,eof
24564,2021-09-02 19:38:41.256,2021-09-02 20:00:07.345,1286.089,0.048,6,192.168.50.248,49320,192.168.50.249,60330,AP,...,0,0,5042690,5042682,5042690,5042690,0,0,0,eof
24565,2021-09-02 10:03:22.901,2021-09-02 10:24:48.348,1285.447,0.000,1,192.168.50.254,0,192.168.50.50,781,0,...,0,0,62287,62286,62287,62287,0,0,0,eof
24566,2021-09-02 08:29:04.239,2021-09-02 08:50:30.350,1286.111,0.023,6,192.168.60.107,52202,192.168.60.101,102,AP,...,0,0,17506851,17506845,17506851,17506851,0,0,0,eof


In [22]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['oct'], mode='markers', name='oct'))
fig_resampled = FigureResampler(fig)
fig_resampled.show()